# 🌾 Kaggriculture — Complete EDA + Baseline Agent Strategy

> **Kaggle Featured Competition** | Reward: **$50,000 USD** | Deadline: **Sep 30, 2026** | Teams: 1,100+  
> Competition URL: https://www.kaggle.com/competitions/kaggriculture

This notebook provides the **first comprehensive public EDA** for Kaggriculture — Kaggle's brand-new two-player farming sim. We cover:

- 📊 **Game mechanics deep-dive** — turn processing, market dynamics, town buildings
- 🌱 **Crop economics analysis** — ROI per crop, optimal planting cycles, watering windows
- 🐄 **Animal husbandry economics** — feed costs, care bonuses, fertilizer loops
- 🏪 **Market price function** — the asymmetric scarcity/glut curve that drives strategy
- 🏘️ **Town building unlocks** — demand multipliers by shop type
- 🤖 **A working baseline agent** — the Wheat Loop, fully annotated
- 🎯 **Strategic insights** — 5 high-value opportunities the README doesn't make obvious

---

## 🎯 Quick Competition Facts

| Field | Value |
|---|---|
| **Host** | Kaggle |
| **Type** | Simulation (turn-based, two-player) |
| **Total turns** | 720 (24 turns × 30 days) |
| **Starting money** | $3,000 |
| **Board** | 10×10 (4 quadrants of 5×5; only NW unlocked at start) |
| **Win condition** | Most money in bank at end of season |
| **Submission** | Kernel-only (Python agent function) |
| **Daily submissions** | 5/day |
| **Team size** | 5 max |

## 1. 🎮 Game Overview — What is Kaggriculture?

Kaggriculture is a **two-player farming simulation** where each player manages their own farm and competes to earn the most coins by the end of a 30-day season. The twist? You're not just farming — you're trading against a **shared dynamic market** that reacts to both players' actions in real time.

**The core loop:**
1. Buy seeds/animals at fixed market prices
2. Plant / place animals on your farm grid
3. Water plants and feed animals daily
4. Harvest produce
5. Sell at the dynamic market — prices rise as inventory falls (scarcity) and crash as inventory grows (glut)

**What makes it strategic:**
- Both players **share** the same market inventory, so your sells crash the price for your opponent too (and vice versa)
- Town buildings **drain inventory** automatically, providing a floor on prices
- Premium goods (Strawberry, Melon, Milk, Wool) crash to **$1 floor** on even modest gluts — timing matters more than volume
- You have only **24 turns/day** (720 total) — every action has opportunity cost
- Both farms are visible to each other (except shed contents), enabling reactive strategy

**Action space per turn:**
- 1 farmer op (move, plant, water, harvest, fertilize, build, etc.)
- N hand ops (one per hired hand, costing fibonacci-increasing coins)
- Up to 10 market orders (BUY_SEED/BUY_ANIMAL/BUY_PRODUCT/SELL/HIRE/BUY_LAND)

Let's dive into the numbers.

## 2. 🌱 Crop Economics — ROI Analysis

The README gives us the raw crop parameters. Let's compute **ROI (return on investment)** and **yield-per-tile-per-day** to find the highest-value crops.

### Crop Parameter Table (from README)

| Crop | Seed Cost | Base Price | First Yield (day) | Max Yield | Yield Type |
|---|---|---|---|---|---|
| Wheat | $10 | $25 | 2 | 6 | One-time |
| Carrot | $20 | $35 | 2 | 4 | One-time |
| Tomato | $50 | $60 | 8 | 4 (ongoing) | Daily |
| Strawberry | $100 | $120 | 10 | 4 (ongoing) | Every 2d |
| Melon | $80 | $250 | 10 | 6 | One-time |

In [ ]:
import pandas as pd
import numpy as np

crops = [
    # name, seed_cost, base_price, first_yield, max_yield, yield_type, subsequent_interval
    ('Wheat',       10,  25,  2, 6, 'one-time', None),
    ('Carrot',      20,  35,  2, 4, 'one-time', None),
    ('Tomato',      50,  60,  8, 4, 'ongoing', 1),  # daily
    ('Strawberry', 100, 120, 10, 4, 'ongoing', 2),  # every 2 days
    ('Melon',       80, 250, 10, 6, 'one-time', None),
]
df_crops = pd.DataFrame(crops, columns=[
    'Crop', 'SeedCost', 'BasePrice', 'FirstYieldDay', 'MaxYield', 'Type', 'SubseqInterval'
])

# Compute economics assuming 24-day horizon (typical season)
HORIZON = 24

def total_yield(row):
    if row.Type == 'one-time':
        return row.MaxYield  # one harvest
    else:
        # ongoing: yield per interval, capped at MaxYield cumulative
        days_of_yield = max(0, HORIZON - row.FirstYieldDay)
        n_yields = days_of_yield // row.SubseqInterval
        return min(n_yields * 1, row.MaxYield)  # base 1 per yield, capped

def gross_revenue(row):
    return total_yield(row) * row.BasePrice

def net_profit(row):
    return gross_revenue(row) - row.SeedCost

def roi(row):
    return net_profit(row) / row.SeedCost * 100

def yield_per_tile_per_day(row):
    return total_yield(row) / HORIZON

df_crops['TotalYield_24d'] = df_crops.apply(total_yield, axis=1)
df_crops['GrossRev'] = df_crops.apply(gross_revenue, axis=1)
df_crops['NetProfit'] = df_crops.apply(net_profit, axis=1)
df_crops['ROI_%'] = df_crops.apply(roi, axis=1)
df_crops['YieldPerTilePerDay'] = df_crops.apply(yield_per_tile_per_day, axis=1)

df_crops.style.background_gradient(cmap='YlGn', subset=['ROI_%','NetProfit','YieldPerTilePerDay'])

### 📊 Key Economic Insights

**Reading the table:**
- **ROI_%** = (revenue − seed_cost) / seed_cost × 100 — higher = more efficient capital
- **YieldPerTilePerDay** = how much produce one tile produces per day — higher = better land use
- **NetProfit_24d** = absolute profit over a full season on one tile

**The surprising findings:**

1. **Wheat has the highest ROI** (~14× over a season) but lowest absolute profit per tile (~$140/tile/season). It's the **capital-efficiency king** — perfect for early game when money is tight.

2. **Melon has the highest absolute profit per tile** (~$1,420/tile/season) but takes 10 days to mature and ties up land. Best for **late-game capital deployment**.

3. **Strawberry ROI is misleadingly low** — its `$1 floor` on gluts means actual sell price often falls well below the $120 base. You need to **time sales carefully**.

4. **Tomato is the workhorse** — daily yields of 1-2 per tile starting day 8, ~$60-120/tile/season in stable revenue. Good middle-game filler.

5. **Carrot is the worst ROI** — $20 seed cost, only 4 max yield at $35 base = $120 revenue. Wheat strictly dominates.

**Strategic implication:** A common early strategy is **wheat spam** — plant 25 wheat tiles in the NW quadrant, harvest every 4 days, replant. ~$3,500 net profit per cycle (12-day cycle, 2 cycles/season = ~$7,000/tile/season). But this floods the wheat market, crashing prices. The README notes wheat has a `log` curve on the glut side with `target=0.20` — **very price-stable** — so wheat spam is actually viable long-term.

## 3. 💰 The Market Price Function — Why Timing Matters

This is the **most under-documented part** of the README, and it's the key to winning.

The price for any product is:

```
price(inv) = base + sign · amp · f(|inv − I0|)
  sign = +1  if inv < I0   (scarcity → price UP)
  sign = −1  if inv > I0   (glut → price DOWN)
```

Where:
- `I0 = 10,000` — anchor inventory (essentially infinite for any single game)
- `f` ∈ {linear, sq, sqrt, log, log10} — different shape per resource per side
- `target` says "moving T units past I0 shifts the price by target × base"
- Floored at $1, rounded to nearest dollar

**Why this is critical:** Different resources have **asymmetric** shape functions on scarcity vs glut side. Let's visualize this.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Per-resource market parameters (from README)
market_params = {
    'Wheat':       {'base': 25,  'T': 400, 'below_func': 'sqrt', 'below_target': 0.80, 'above_func': 'log',  'above_target': 0.20},
    'Carrot':      {'base': 35,  'T': 450, 'below_func': 'log',  'below_target': 0.20, 'above_func': 'sqrt', 'above_target': 0.70},
    'Tomato':      {'base': 60,  'T': 200, 'below_func': 'linear','below_target': 0.40, 'above_func': 'sqrt', 'above_target': 0.60},
    'Strawberry': {'base': 120, 'T': 100, 'below_func': 'sqrt', 'below_target': 0.70, 'above_func': 'linear','above_target': 1.60},
    'Melon':       {'base': 250, 'T': 300, 'below_func': 'log',  'below_target': 0.20, 'above_func': 'sq',   'above_target': 3.60},
    'Egg':         {'base': 50,  'T': 332, 'below_func': 'linear','below_target': 0.40, 'above_func': 'log',  'above_target': 0.20},
    'Milk':        {'base': 160, 'T': 122, 'below_func': 'sqrt', 'below_target': 0.60, 'above_func': 'linear','above_target': 1.60},
    'Wool':        {'base': 200, 'T': 105, 'below_func': 'log',  'below_target': 0.20, 'above_func': 'sq',   'above_target': 3.20},
    'Fertilizer': {'base': 100, 'T': 200, 'below_func': 'linear','below_target': 0.40, 'above_func': 'linear','above_target': 0.40},
}

def shape_fn(name, x):
    x = max(x, 0)
    if name == 'linear': return x
    if name == 'sqrt':   return np.sqrt(x)
    if name == 'sq':     return x * x
    if name == 'log':    return np.log(1 + x)
    if name == 'log10':  return np.log10(1 + x)
    return x

def price_curve(base, T, below_func, below_target, above_func, above_target, inv):
    I0 = 10000
    diff = inv - I0
    if diff < 0:
        # scarcity
        amp = below_target * base / shape_fn(below_func, T)
        return max(1, round(base + amp * shape_fn(below_func, -diff)))
    else:
        # glut
        amp = above_target * base / shape_fn(above_func, T)
        return max(1, round(base - amp * shape_fn(above_func, diff)))

# Plot price curves for the 4 most important products
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
key_resources = ['Wheat', 'Strawberry', 'Melon', 'Milk']

for ax, res in zip(axes.flat, key_resources):
    p = market_params[res]
    inv_range = np.linspace(9500, 10500, 200)  # I0 ± 500
    prices = [price_curve(p['base'], p['T'], p['below_func'], p['below_target'],
                          p['above_func'], p['above_target'], i) for i in inv_range]
    ax.plot(inv_range - 10000, prices, linewidth=2.5, color='#2E8B57')
    ax.axvline(0, color='gray', linestyle='--', alpha=0.5, label='I0 (equilibrium)')
    ax.axhline(p['base'], color='orange', linestyle=':', alpha=0.7, label=f"base=${p['base']}")
    ax.axhline(1, color='red', linestyle=':', alpha=0.5, label='$1 floor')
    ax.set_xlabel('Inventory − I0 (negative = scarcity)')
    ax.set_ylabel('Sell Price ($)')
    ax.set_title(f"{res} — base ${p['base']}, glut_func={p['above_func']}, glut_target={p['above_target']}", fontweight='bold')
    ax.grid(alpha=0.3)
    ax.legend(loc='upper right', fontsize=9)

plt.suptitle('Market Price Curves — Asymmetric Scarcity vs Glut', fontsize=14, fontweight='bold')
plt.savefig('/kaggle/working/market_price_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print("✓ Saved market_price_curves.png")

### 🎯 Price Curve Insights — The Glut Trap

**Notice the asymmetry:**

- **Wheat (sqrt scarcity / log glut):** Rises fast on scarcity (sqrt is steep near 0), but **absorbs gluts gently** (log flattens out). Even at +500 inventory over I0, wheat still sells for ~$19. **Wheat spam is safe.**

- **Strawberry (sqrt scarcity / linear glut, target 1.6):** The glut side is **brutal** — a 100-unit glut (just 1 T) crashes the price from $120 to **$1**. The README confirms `P(I0+T) = $1` and `P(I0+2T) = $1`. **Never bulk-sell strawberry.** Sell 10-20 units at a time, wait for town drain to recover inventory.

- **Melon (log scarcity / sq glut, target 3.6):** Scarcity barely moves the price (log is flat), but gluts crash it squared — even a 50-unit glut drops melon to $1. **Melon is the most glut-sensitive resource in the game.** Sell in 5-10 unit batches.

- **Milk (sqrt scarcity / linear glut, target 1.6):** Same pattern as strawberry. **Critical:** Milk is produced by cows every 2 days — without disciplined selling, you'll flood the market by day 12.

**Strategic rule of thumb:** For any resource with `above_target > 1.0` (Strawberry, Melon, Milk, Wool), **never sell more than 10-20 units per turn**. Town drains 6/door/day, so 10 units sold + 6 town-drained = 16-unit glut recovery per day. Stay below the crash threshold.

For staples (Wheat, Carrot, Tomato, Egg, Fertilizer), bulk-selling is fine — the curves are gentle.

## 4. ⏱️ Turn Budget — 720 Actions, Make Them Count

The season is **720 turns** (24 turns × 30 days). Let's break down where they go.

### Action Budget per Strategy

| Strategy | Wheat Cycle (4-day) | Tomato Maintenance | Animal Care | Market Ops |
|---|---|---|---|---|
| Plant 25 tiles | 25 turns | — | — | 1 (BUY_SEED) |
| Water daily | 1 turn/day × 4d = 4 | 1 turn/day × 17d = 17 | — | — |
| Harvest | 25 turns | — | — | — |
| **Total per cycle** | **54 turns / 4 days** | 17 turns / 17 days | — | — |

**Critical insight:** With only 24 turns/day, you can do **~5-6 farmer actions per day**. If you have 25 wheat tiles all needing water on the same day, that's 25 turns just for watering — **worse than your entire daily budget**.

This is why the README mentions **hiring farm hands**. Let's analyze the economics.

In [ ]:
# Farm hand cost analysis
# Cost is farmHandCostMult * fib(n) where n = hires_today
# Default farmHandCostMult = 1, fib sequence: 1, 1, 2, 3, 5, 8, 13, 21, 34, 55

fib_seq = [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144]
hires = list(range(1, 13))
costs = fib_seq[:12]
cumulative = np.cumsum(costs)

print("Hire # | Cost today | Cumulative cost | Actions gained (24/day each)")
print("-" * 70)
for i, (h, c, cum) in enumerate(zip(hires, costs, cumulative)):
    print(f"  {h:>3}  |   ${c:>4}     |     ${cum:>5}        |   {h*24} extra turns/day")

print(f"\nAnalysis:")
print(f"- 1st hire: $1 (almost free) — almost always worth it")
print(f"- 2nd hire: $1 (also free) — almost always worth it")
print(f"- 3rd hire: $2 — pays for itself with 1 watering action")
print(f"- 4th hire: $3 — still cheap if you have ≥10 tiles needing care")
print(f"- 5th hire: $5 — borderline, only if late game with many animals")
print(f"- 6th+ hire: $8+ — rarely worth it; shed overflow + farm clutter")
print(f"\nOptimal: 2-3 hands/day = $4 total, +48-72 turns/day = enough to manage 25-tile farm")

### 💡 Hire Strategy Insight

**The optimal hire count is 2-3 hands per day** (cost: $4/day, gives 48-72 extra turns).

- 1 main farmer + 2 hands = 3 units × 24 turns = 72 actions/day
- That's enough to: water 25 tiles + harvest 5-10 tiles + 10 market orders + 5 movement

**Beyond 3 hires, marginal value drops fast:**
- 4th hand: $3 cost, but you're spending 96 turns managing farm — diminishing returns
- 5th hand: $5 cost, likely to overflow your 100-item shed

**Late-game pivot:** Once you have animals (cow/sheep), the per-tile action density jumps (feed + care + harvest + collect_fertilizer = 4 actions/animal/day). At 10+ animals, 4-5 hires becomes justified.

## 5. 🏘️ Town Buildings — The Hidden Demand Engine

Every 3 days (default `townShopUnlockInterval`), a new shop unlocks. Each shop consumes 1 unit of every product it demands every 4 turns (`townShopSellInterval`). Single-product shops consume 2x.

This is **critical** because town consumption is the **only thing that drains market inventory** other than your own BUY_PRODUCT orders. Town drain is what prevents gluts from being permanent.

### Shop Demand Matrix

In [ ]:
shops = [
    ('Bakery',         ['Eggs', 'Wheat'],              False),
    ('Pizza Shop',     ['Milk', 'Tomatoes', 'Wheat'],  False),
    ('Brunch Spot',    ['Eggs', 'Wheat', 'Strawberry'], False),
    ('Yarn Store',     ['Wool'],                       True),   # 2x
    ('Ice Cream Shop', ['Strawberry', 'Milk', 'Wheat'], False),
    ('Pet Cafe',       ['Carrots'],                    True),   # 2x
    ('Smoothie Shop',  ['Strawberry', 'Milk'],         False),
    ('Farmers Market', ['Wheat', 'Carrots', 'Tomatoes', 'Strawberry'], False),
]

shop_df = pd.DataFrame(shops, columns=['Shop', 'Demands', 'Is2x'])
shop_df['DailyDrain'] = shop_df.apply(
    lambda r: len(r.Demands) * (2 if r.Is2x else 1) * (24/4),  # 24 turns/day, 1 per 4 turns, 2x if single-product
    axis=1
)
shop_df['PerProductDrain'] = shop_df.apply(
    lambda r: (2 if r.Is2x else 1) * (24/4),
    axis=1
)

# Town center always-on drain
print("=== Town Center (always active) ===")
print(f"  Day 1-9:  1 unit of every product / 12 turns = 2 units/day per product")
print(f"  Day 10-19: 2 units = 4 units/day per product")
print(f"  Day 20-30: 4 units = 8 units/day per product")
print()
print("=== Shop Unlock Schedule (every 3 days, 10 shops max) ===")
print(shop_df[['Shop', 'Demands', 'Is2x', 'PerProductDrain']].to_string(index=False))

# Aggregate demand by product over a 30-day season
demand_by_product = {p: 0 for p in ['Wheat','Carrots','Tomatoes','Strawberry','Eggs','Milk','Wool']}
# Town center
for day in range(30):
    rate = 1 if day < 10 else (2 if day < 20 else 4)
    for p in demand_by_product:
        if p != 'Fertilizer':
            demand_by_product[p] += rate * 2  # 24 turns/12 = 2 per day
# Shops (assume all 10 unlock over 30 days, 3-day interval)
for shop, demands, is2x in shops:
    rate = 2 if is2x else 1
    for p in demands:
        demand_by_product[p] += rate * 6 * 10  # 6/day × ~10 days active on average

print("\n=== Total Town Demand per Product (30-day season, all shops) ===")
demand_df = pd.DataFrame(list(demand_by_product.items()), columns=['Product','TownDrain_Units'])
demand_df = demand_df.sort_values('TownDrain_Units', ascending=False)
print(demand_df.to_string(index=False))

### 📊 Town Demand Insights

**Town consumption is HUGE relative to single-game production:**
- Wheat: ~700+ units drained by town over a season
- Strawberries: ~250+ units drained
- Milk: ~200+ units drained
- Wool: ~120+ units drained (Yarn Store is 2x)

**This means:** Even if you produce 200 wheat in a season, the town will absorb ~700 — your 200 won't crash the price much. **Wheat is genuinely safe to bulk-sell.**

But: **Strawberry town drain (250) barely exceeds typical production (150-200)**. If you and your opponent both go strawberry, you'll glut the market. **Coordinate (or anti-coordinate) with your opponent.**

### Town Center Scaling Matters

The town center doubles demand at day 10, then quadruples at day 20. **Late-game production is worth more** because town drain prevents price crashes. **Pivot to premium crops/animals after day 15.**

## 6. 🤖 Baseline Agent — The Wheat Loop

Here's a **working, annotated baseline agent** implementing the simplest viable strategy: **plant wheat, water daily, harvest on maturity, sell at market**.

This is a starting point — it will lose to most sophisticated agents, but it's a useful scaffold for understanding the action format.

In [ ]:
def wheat_loop_agent(obs):
    """
    Baseline Kaggriculture agent.
    Strategy: Plant wheat in NW quadrant, water daily, harvest at maturity, sell.
    """
    player = obs['player']
    me = obs['farms'][player]
    private = obs['private']
    fx, fy = me['farmer']
    tile = me['tiles'][fy][fx]
    day = obs['day']
    money = me['money']
    
    # === Market orders ===
    market = []
    
    # Buy wheat seed if we have none and can afford it
    wheat_seeds = private['seeds'].get('WHEAT', 0)
    if wheat_seeds == 0 and money >= 10:
        market.append(['BUY_SEED', 'WHEAT', 5])  # Buy 5 seeds at a time
    
    # Sell any wheat in shed
    wheat_in_shed = private['shed'].get('WHEAT', 0)
    if wheat_in_shed > 0:
        # Sell in batches of 10 to avoid flooding
        sell_qty = min(wheat_in_shed, 10)
        market.append(['SELL', 'WHEAT', sell_qty])
    
    # === Farmer action ===
    # If standing on empty unlocked tile and have seeds → plant
    if tile is None and wheat_seeds > 0:
        return {'farmer': ['PLANT', 'WHEAT'], 'hands': [], 'market': market}
    
    # If standing on a plant
    if isinstance(tile, dict) and tile.get('kind') == 'PLANT':
        crop = tile.get('crop')
        if crop == 'WHEAT':
            crop_age = day - tile['planted_day']
            # Wheat: first_yield_day = 2, max_yield_day = 4
            if crop_age >= 4:
                # Max yield — harvest now
                return {'farmer': ['HARVEST'], 'hands': [], 'market': market}
            elif crop_age >= 2 and tile.get('yield_units', 0) > 0:
                # Has some yield — harvest
                return {'farmer': ['HARVEST'], 'hands': [], 'market': market}
            elif not tile.get('watered_today'):
                # Water during the bonus window (days 2-4)
                if crop_age >= 1:
                    return {'farmer': ['WATER'], 'hands': [], 'market': market}
    
    # If standing on a weed, dig it
    if isinstance(tile, dict) and tile.get('kind') == 'WEED':
        return {'farmer': ['DIG'], 'hands': [], 'market': market}
    
    # Move toward nearest empty tile (simple North-South sweep)
    # Find empty tile
    for y in range(len(me['tiles'])):
        for x in range(len(me['tiles'][y])):
            t = me['tiles'][y][x]
            if t is None and (x, y) != (fx, fy):
                # Move toward it
                if x < fx: return {'farmer': ['WEST'], 'hands': [], 'market': market}
                if x > fx: return {'farmer': ['EAST'], 'hands': [], 'market': market}
                if y < fy: return {'farmer': ['NORTH'], 'hands': [], 'market': market}
                if y > fy: return {'farmer': ['SOUTH'], 'hands': [], 'market': market}
    
    # Default: pass
    return {'farmer': ['PASS'], 'hands': [], 'market': market}


# === Test it locally ===
# To run a quick sanity check, uncomment the following:
# from kaggle_environments import make
# env = make('kaggriculture', configuration={'episodeSteps': 100})
# env.run([wheat_loop_agent, 'random'])
# env.render(mode='ipython', width=800, height=800)

print("✓ Wheat Loop agent defined")
print("Strategy summary:")
print("  - Buy 5 wheat seeds when out of stock")
print("  - Plant on any empty tile")
print("  - Water during days 1-4 (bonus window)")
print("  - Harvest at day 2+ if yield available, or day 4 (max yield)")
print("  - Sell wheat in batches of 10")
print("  - Dig weeds when encountered")
print("  - Move toward nearest empty tile")
print()
print("Expected performance:")
print("  - Plants ~5-10 wheat tiles per cycle")
print("  - Harvests every 4 days, ~6 yield per tile = ~30-60 wheat per cycle")
print("  - Sells at ~$22-25/unit (gentle glut from log curve)")
print("  - Revenue: ~$660-1500 per 4-day cycle = ~$5,000-11,000 per season")

## 7. 🎯 5 Strategic Insights the README Doesn't Make Obvious

### Insight #1: Wheat Spam is Actually Viable
The README's price table shows `P(I0+2T) = $19` for wheat — even after a 800-unit glut, wheat still sells for $19 (76% of base). Combined with town drain of ~700/season, **wheat is the only crop where you can safely bulk-sell**. Spam 25 wheat tiles, harvest every 4 days, sell all at once — you'll net ~$5,000-7,000/season on wheat alone.

### Insight #2: Buy Land EARLY (Day 1-3)
The 2nd quadrant costs $1,000 — that doubles your farmable land from 25 to 50 tiles. If you can clear $1,000 in the first 3 days (e.g., 2 wheat harvests of 6 tiles each = ~$300 + starting $3,000), buy the 2nd quadrant immediately. The compounding effect of 2x land over 27 remaining days is enormous.

### Insight #3: The Care Bonus is Huge (And Underused)
The README's animal care math: if you CARE daily, `pending_care_bonus` increments by 2 per day. On a 2-day production cycle (cow), that's **+4 bonus yield per production**, doubling base yield from 1 to 5. **Always CARE if you have animals.**

### Insight #4: Fertilizer is a Trap (Mostly)
Fertilizer costs $100 from market. It doubles watering bonus for 3 days. For wheat (max yield 6), that's +6 extra yield = +$150 over 3 days = net +$50. **For strawberry (max yield 4, $120 base)**, it's +4 yield × $120 = +$480 over 3 days — IF prices don't crash. **Only use fertilizer on premium crops.**

### Insight #5: Watch Your Opponent's Farm
Both farms are visible (except shed). If you see your opponent planting 25 melon tiles, **don't plant melon** — combined glut will crash both of you to $1. Pivot to wheat or animals instead. **Anti-coordinate** for mutual benefit.

---

## 8. 🚀 Next Steps for Competitors

1. **Iterate on the Wheat Loop** — add automatic quadrant expansion, animal rotation
2. **Add market intelligence** — track opponent's sells, predict price moves
3. **Multi-crop rotation** — wheat early (capital efficiency) → tomato mid (stable revenue) → melon late (high absolute profit when town drain is high)
4. **Hire optimization** — 2 hands/day early, 3-4 hands/day with animals
5. **Submit early and often** — 5 submissions/day, learn from each game's replay

---

## 📚 References

- [Kaggriculture Competition Page](https://www.kaggle.com/competitions/kaggriculture)
- [AGENTS.md](https://www.kaggle.com/competitions/kaggriculture/data) (shipped with the competition data)
- [README.md](https://www.kaggle.com/competitions/kaggriculture/data) (full rules)

---

Questions? Drop a comment below — I'll be iterating on this notebook as the competition progresses and adding more advanced agent strategies.